In [2]:
import re
import numpy as np
import pandas as pd

df = pd.read_csv('training_data_ready.csv')

### 2. Berechnung der Abweichungen pro Messpunkt
Da die Spalten `A_1_center`, `A_1_front`, etc. den gleichen Messpunkt an verschiedenen Positionen repräsentieren, berechnen wir nun für jeden der 7 Messpunkte (A_1 bis A_7) die **Standardabweichung (std)** und die **Spannweite (range, Max-Min)** über die 5 Positionen. Diese Metriken sind ein direktes Maß für die Abweichungen der Messwerte zueinander.

In [ ]:
messpunkte = ['A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7']
positionen = ['center', 'front', 'left', 'right', 'back']

alle_spalten = [f"{mp}_{pos}" for mp in messpunkte for pos in positionen]
vorhandene_spalten = [col for col in alle_spalten if col in df.columns]

stats_list = []
for col in vorhandene_spalten:
    c_min = df[col].min()
    c_max = df[col].max()
    c_std = df[col].std()
    c_range = c_max - c_min
    stats_list.append({
        'Sensor_Position': col,
        'Min': c_min,
        'Max': c_max,
        'Range': c_range,
        'Std': c_std
    })

stats_df = pd.DataFrame(stats_list).set_index('Sensor_Position')
print("Statistische Auswertung für die 35 einzelnen Sensor-Positionen:")
display(stats_df)


Statistische Auswertung für die 35 einzelnen Sensor-Positionen:


,Min,Max,Range,Std
Sensor_Position,,,,
A_1_center,0.2834,0.7123,0.4289,0.087470
A_1_front,-0.3819,1.0144,1.3963,0.413486
A_1_left,0.1567,0.7473,0.5906,0.106633
A_1_right,-0.2204,1.0109,1.2313,0.341617
A_1_back,-0.1801,0.3655,0.5456,0.103370
A_2_center,5.8844,8.3840,2.4996,0.571027
A_2_front,6.3864,9.4911,3.1047,0.736743
A_2_left,5.6156,7.8971,2.2815,0.501896
A_2_right,5.2000,7.0277,1.8277,0.370886


In [ ]:
df['Objektname'] = df['source_file'].apply(lambda x: x[:x.rfind('_')] if '_' in x else x)

obj_std = df.groupby('Objektname')[vorhandene_spalten].std()

min_std = obj_std.min()
max_std = obj_std.max()

objekt_analyse_df = pd.DataFrame({
    'Min_Std_Objekt': min_std,
    'Max_Std_Objekt': max_std
})

print("Minimale und maximale Standardabweichung eines Objekts pro Sensor-Position:")
display(objekt_analyse_df)


Minimale und maximale Standardabweichung eines Objekts pro Sensor-Position:


,Min_Std_Objekt,Max_Std_Objekt
A_1_center,0.001344,0.101353
A_1_front,0.104723,0.681606
A_1_left,0.021401,0.171745
A_1_right,0.043731,0.553240
A_1_back,0.016509,0.195707
A_2_center,0.009263,0.124422
A_2_front,0.046293,0.237588
A_2_left,0.010182,0.177717
A_2_right,0.028000,0.229858
A_2_back,0.002515,0.178051


In [ ]:
df['Objektname'] = df['source_file'].apply(lambda x: x[:x.rfind('_')] if '_' in x else x)

obj_std = df.groupby('Objektname')[vorhandene_spalten].std()

min_std = obj_std.min()
max_std = obj_std.max()

df_obj_mean = df.groupby('Objektname')[vorhandene_spalten].transform('mean')

df_dev = (df[vorhandene_spalten] - df_obj_mean).abs()

max_dev_from_mean = df_dev.max()

df_rel_dev = df_dev.div(df_obj_mean.abs().replace(0, np.nan))

max_rel_dev_pct = df_rel_dev.max() * 100

count_gt_10pct = (df_rel_dev > 0.25).sum()
count_gt_25pct = (df_rel_dev > 0.50).sum()
count_gt_50pct = (df_rel_dev > 1.00).sum()

objekt_analyse_df = pd.DataFrame({
    'Min_Std_Objekt': min_std,
    'Max_Std_Objekt': max_std,
    'Max_Abw_vom_Mittelwert_Absolut': max_dev_from_mean,
    'Max_Abw_vom_Mittelwert_Relativ_%': max_rel_dev_pct,
    'Anzahl_Ausreisser_>_25%_Relativ': count_gt_10pct,
    'Anzahl_Ausreisser_>_50%_Relativ': count_gt_25pct,
    'Anzahl_Ausreisser_>_100%_Relativ': count_gt_50pct
})

print("Erweiterte Analyse pro Sensor-Position (mit relativen Ausreißern):")
display(objekt_analyse_df)


Erweiterte Analyse pro Sensor-Position (mit relativen Ausreißern):


,Min_Std_Objekt,Max_Std_Objekt,Max_Abw_vom_Mittelwert_Absolut,Max_Abw_vom_Mittelwert_Relativ_%,Anzahl_Ausreisser_>_25%_Relativ,Anzahl_Ausreisser_>_50%_Relativ,Anzahl_Ausreisser_>_100%_Relativ
A_1_center,0.001344,0.101353,0.194100,42.140686,4,0,0
A_1_front,0.104723,0.681606,0.837333,647.954356,152,121,63
A_1_left,0.021401,0.171745,0.253067,61.082455,33,5,0
A_1_right,0.043731,0.553240,0.638567,284.082434,140,109,62
A_1_back,0.016509,0.195707,0.253062,7866.666667,146,120,78
A_2_center,0.009263,0.124422,0.182400,2.651245,0,0,0
A_2_front,0.046293,0.237588,0.302933,4.374699,0,0,0
A_2_left,0.010182,0.177717,0.289014,3.790444,0,0,0
A_2_right,0.028000,0.229858,0.447700,6.599838,0,0,0
A_2_back,0.002515,0.178051,0.301500,3.936494,0,0,0


In [ ]:
df['Objektname'] = df['source_file'].apply(lambda x: x[:x.rfind('_')] if '_' in x else x)

obj_std = df.groupby('Objektname')[vorhandene_spalten].std()
min_std = obj_std.min()
max_std = obj_std.max()

q1 = df.groupby('Objektname')[vorhandene_spalten].transform(lambda x: x.quantile(0.25))
q3 = df.groupby('Objektname')[vorhandene_spalten].transform(lambda x: x.quantile(0.75))
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

is_outlier_lower = df[vorhandene_spalten] < lower_bound
is_outlier_upper = df[vorhandene_spalten] > upper_bound
is_outlier = is_outlier_lower | is_outlier_upper

outlier_counts = is_outlier.sum()

dist_to_q1 = (q1 - df[vorhandene_spalten]).where(is_outlier_lower, np.nan)
dist_to_q3 = (df[vorhandene_spalten] - q3).where(is_outlier_upper, np.nan)

outlier_distances = dist_to_q1.fillna(dist_to_q3)

iqr_safe = iqr.replace(0, np.nan)

outlier_factors = outlier_distances / iqr_safe

avg_outlier_factor = outlier_factors.mean()

objekt_analyse_df = pd.DataFrame({
    'Min_Std_Objekt': min_std,
    'Max_Std_Objekt': max_std,
    'Anzahl_Ausreisser_IQR': outlier_counts,
    'Durchschn_Faktor_Ausreisser_zu_IQR': avg_outlier_factor
})

print("Zusammengefasste Ausreißer-Analyse pro Sensor-Position (IQR-Methode):")
display(objekt_analyse_df)


Zusammengefasste Ausreißer-Analyse pro Sensor-Position (IQR-Methode):


,Min_Std_Objekt,Max_Std_Objekt,Anzahl_Ausreisser_IQR,Durchschn_Faktor_Ausreisser_zu_IQR
A_1_center,0.001344,0.101353,14,6.400639
A_1_front,0.104723,0.681606,3,4.686378
A_1_left,0.021401,0.171745,4,1.936348
A_1_right,0.043731,0.553240,4,2.068866
A_1_back,0.016509,0.195707,1,4.000000
A_2_center,0.009263,0.124422,3,2.148213
A_2_front,0.046293,0.237588,5,8.392153
A_2_left,0.010182,0.177717,7,1.969796
A_2_right,0.028000,0.229858,4,1.984039
A_2_back,0.002515,0.178051,7,2.315596


In [ ]:
outlier_rows_mask = is_outlier.any(axis=1)

if outlier_rows_mask.any():
    info_cols = ['source_file', 'Objektname']
    info_cols = [col for col in info_cols if col in df.columns]

    detailed_outliers_df = df.loc[outlier_rows_mask, info_cols].copy()

    outlier_bools = is_outlier.loc[outlier_rows_mask]

    affected_sensors = outlier_bools.apply(lambda row: row.index[row].tolist(), axis=1)

    detailed_outliers_df['Anzahl_Ausreisser_Sensoren'] = affected_sensors.apply(len)
    detailed_outliers_df['Betroffene_Sensoren'] = affected_sensors.apply(lambda x: ', '.join(x))

    def get_outlier_values_str(row_idx):
        sensors = affected_sensors.loc[row_idx]
        return ', '.join([f"{sensor}: {df.loc[row_idx, sensor]:.4f}" for sensor in sensors])
        
    detailed_outliers_df['Ausreisser_Werte'] = [get_outlier_values_str(idx) for idx in detailed_outliers_df.index]

    detailed_outliers_df = detailed_outliers_df.sort_values(by='Anzahl_Ausreisser_Sensoren', ascending=False)
    
    print(f"Insgesamt wurden {len(detailed_outliers_df)} Datenpunkte (Zeilen) mit mindestens einem Ausreißer gefunden.")
    display(detailed_outliers_df.head(50))
else:
    print("Es wurden keine Ausreißer gefunden.")


Insgesamt wurden 101 Datenpunkte (Zeilen) mit mindestens einem Ausreißer gefunden.


,source_file,Objektname,Anzahl_Ausreisser_Sensoren,Betroffene_Sensoren,Ausreisser_Werte
124,6eck_L_L_1.csv,6eck_L_L,7,"A_1_left, A_2_left, A_2_back, A_4_front, A_4_l...","A_1_left: 0.2689, A_2_left: 6.0089, A_2_back: ..."
100,Lumi_V_L_L_3.csv,Lumi_V_L_L,6,"A_1_left, A_3_left, A_3_back, A_4_center, A_5_...","A_1_left: 0.2480, A_3_left: -0.6171, A_3_back:..."
73,Lumi_L_L_1.csv,Lumi_L_L,5,"A_3_front, A_4_center, A_5_left, A_7_left, A_7...","A_3_front: 0.1723, A_4_center: -4.6620, A_5_le..."
95,Lumi_V_L_U_4.csv,Lumi_V_L_U,5,"A_1_back, A_2_right, A_2_back, A_3_back, A_4_c...","A_1_back: -0.0902, A_2_right: 5.7561, A_2_back..."
43,Rechteck_L_L_3.csv,Rechteck_L_L,5,"A_1_front, A_2_right, A_4_left, A_5_front, A_7...","A_1_front: -0.2317, A_2_right: 5.4577, A_4_lef..."
151,Durch_L_L_D_O_6.csv,Durch_L_L_D_O,5,"A_3_back, A_4_back, A_5_back, A_6_left, A_6_back","A_3_back: 0.8657, A_4_back: -4.4271, A_5_back:..."
20,3eck_L_U_1.csv,3eck_L_U,4,"A_2_left, A_2_back, A_4_back, A_6_right","A_2_left: 7.3358, A_2_back: 7.4195, A_4_back: ..."
76,Lumi_L_L_4.csv,Lumi_L_L,4,"A_3_front, A_6_front, A_7_center, A_7_left","A_3_front: 0.1726, A_6_front: 1.9657, A_7_cent..."
82,Lumi_L_L_10.csv,Lumi_L_L,4,"A_3_left, A_6_front, A_6_left, A_7_left","A_3_left: -0.5338, A_6_front: 1.8400, A_6_left..."
40,Rechteck_L_U_7.csv,Rechteck_L_U,4,"A_1_center, A_2_back, A_5_center, A_6_right","A_1_center: 0.4483, A_2_back: 6.2162, A_5_cent..."
